# Hyperparameter Tuning — Vanilla LSTM on CMAPSS

This notebook performs random search over five hyperparameters of the
Vanilla LSTM architecture using **engine-grouped 5-fold cross-validation**.
Output is a CSV log of all evaluated configurations plus a final retrained
best-config model evaluated on the held-out test set.

## Methodology

- **Search strategy**: random search over 30 configurations (Bergstra & Bengio,
  2012, find this superior to grid search for neural-network HPO).
- **Evaluation**: 5-fold CV with `GroupKFold` over engine IDs, mirroring the
  same engine-grouped split used in `Vanilla_LSTM.ipynb`. The same engine
  is never split between train and validation within a fold.
- **Objective**: mean validation RMSE across the 5 folds (lower is better).
- **Reporting**: best config retrained at full 100 epochs on the combined
  fold-train data, evaluated once on the test set for RMSE and NASA score.

## Search space

| Hyperparameter | Distribution                | Notes |
|---|---|---|
| `hidden_size`   | Uniform from {32, 64, 128, 256} | Capacity |
| `num_layers`    | Uniform from {1, 2}             | Depth   |
| `dropout`       | Uniform [0.0, 0.5] (only if `num_layers ≥ 2`) | Regularisation |
| `learning_rate` | **Log-uniform** [1e-4, 1e-2]    | Step size |
| `batch_size`    | Uniform from {64, 128, 256, 512} | Optimisation |

## Resumability

Each fold result is appended to a CSV after completion. Re-running the
search loop reads the CSV and skips already-evaluated `(config_id, fold)`
pairs. This protects against Colab session timeouts and lets the search
be split across multiple runs without losing progress.


## 1. Configuration

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────

PREPROCESSED_DIR = 'preprocessed/'        # output folder from preprocessing.ipynb
RESULTS_DIR      = 'tuning_results/'      # CSV logs of search results
MODELS_DIR       = 'saved_models/'        # weights of the best retrained model

# Which dataset are we tuning on?
TUNE_DS          = 'FD001'

# Reproducibility
SEED             = 42

# Search budget
N_CONFIGS        = 30                     # random configurations to evaluate
N_FOLDS          = 5                      # GroupKFold engine-grouped CV
SEARCH_EPOCHS    = 50                     # max epochs per fold during search
EARLY_STOP_PATIENCE = 10                  # epochs without val improvement -> stop

# Final retrain settings (uses single 90/10 engine-grouped split, like main notebook)
FINAL_EPOCHS         = 100
FINAL_PATIENCE       = 15
FINAL_VAL_FRACTION   = 0.10

# ──────────────────────────────────────────────────────────────────────────────


## 2. Imports and Reproducibility

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import random
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset

from sklearn.model_selection import GroupShuffleSplit, GroupKFold


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"PyTorch       : {torch.__version__}")
print(f"Device        : {DEVICE}")
print(f"Seed          : {SEED}")
print(f"Tuning on     : {TUNE_DS}")
print(f"Configs       : {N_CONFIGS}  ({N_FOLDS}-fold CV each)")


PyTorch       : 2.10.0+cpu
Device        : cpu
Seed          : 42
Tuning on     : FD001
Configs       : 30  (5-fold CV each)


## 3. Load Preprocessed Data

Loads the same `.npy` arrays produced by `preprocessing.ipynb`, including
`engine_ids_train.npy` which is required for the engine-grouped split.


In [3]:
class CMAPSSDataset(Dataset):
    """Same as in Vanilla_LSTM.ipynb — wraps numpy arrays as a torch Dataset."""
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def load_arrays(preprocessed_dir, dataset):
    """Returns dict with X_train, y_train, engine_ids, X_test, y_test, feature_cols."""
    p = os.path.join(preprocessed_dir, dataset)
    return {
        'X_train':      np.load(f'{p}_X_train.npy'),
        'y_train':      np.load(f'{p}_y_train.npy'),
        'engine_ids':   np.load(f'{p}_engine_ids_train.npy'),
        'X_test':       np.load(f'{p}_X_test.npy'),
        'y_test':       np.load(f'{p}_y_test.npy'),
        'feature_cols': list(np.load(f'{p}_feature_cols.npy')),
    }


arrays = load_arrays(PREPROCESSED_DIR, TUNE_DS)
N_FEATURES = arrays['X_train'].shape[2]
WINDOW_SIZE = arrays['X_train'].shape[1]

print(f"Dataset      : {TUNE_DS}")
print(f"X_train      : {arrays['X_train'].shape}")
print(f"X_test       : {arrays['X_test'].shape}")
print(f"Engines      : {len(np.unique(arrays['engine_ids']))} (train), "
      f"{len(arrays['X_test'])} (test)")
print(f"n_features   : {N_FEATURES}")
print(f"window_size  : {WINDOW_SIZE}")


Dataset      : FD001
X_train      : (17731, 30, 15)
X_test       : (100, 30, 15)
Engines      : 100 (train), 100 (test)
n_features   : 15
window_size  : 30


## 4. Model and Training Functions

Self-contained copies of the Vanilla LSTM and training functions, parameterised
by hyperparameter dict so they can be invoked per-config without touching the
main notebook. Includes early stopping, which the production notebook does not
need (its training is short and deterministic).


In [4]:
class VanillaLSTM(nn.Module):
    """
    Same architecture as the production notebook, with two configurable knobs
    exposed for the search: hidden_size and num_layers (and dropout when
    num_layers >= 2). PyTorch's nn.LSTM applies dropout *between* stacked
    layers, so it has no effect when num_layers == 1.
    """
    def __init__(self, input_size, hidden_size=100, num_layers=1, dropout=0.0):
        super().__init__()
        # PyTorch warns if dropout > 0 and num_layers == 1; silence by clamping.
        effective_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            bias        = True,
            batch_first = True,
            dropout     = effective_dropout,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.fc(last)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X).squeeze(-1)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
    return total / len(loader.dataset)


def validate(model, loader, criterion):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X).squeeze(-1)
            total += criterion(pred, y).item() * len(y)
    return total / len(loader.dataset)


def train_with_early_stopping(model, train_loader, val_loader,
                              learning_rate, max_epochs, patience):
    """
    Train until val MSE stops improving for `patience` epochs, or max_epochs.
    Returns (best_val_mse, n_epochs_run, best_state_dict).
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    best_val_mse  = float('inf')
    best_state    = None
    epochs_no_imp = 0

    for epoch in range(1, max_epochs + 1):
        train_one_epoch(model, train_loader, optimizer, criterion)
        val_mse = validate(model, val_loader, criterion)

        if val_mse < best_val_mse:
            best_val_mse  = val_mse
            best_state    = {k: v.detach().clone().cpu()
                             for k, v in model.state_dict().items()}
            epochs_no_imp = 0
        else:
            epochs_no_imp += 1
            if epochs_no_imp >= patience:
                return best_val_mse, epoch, best_state

    return best_val_mse, max_epochs, best_state


## 5. Random Search Sampler

Each `config` is a self-describing dictionary so it can be serialised to CSV
and reloaded during analysis. We seed the sampler so the same `N_CONFIGS`
configurations are produced every time — the search itself is deterministic
even though we call it "random" search.


In [5]:
def sample_configs(n_configs, seed=SEED):
    """
    Generate a deterministic list of `n_configs` random hyperparameter dicts.
    Re-running with the same seed yields the same list — so a search that was
    interrupted partway can be resumed with config_id stable across sessions.
    """
    rng = np.random.default_rng(seed)

    HIDDEN_CHOICES = [32, 64, 128, 256]
    LAYER_CHOICES  = [1, 2]
    BATCH_CHOICES  = [64, 128, 256, 512]
    LR_LOG_LO, LR_LOG_HI = np.log10(1e-4), np.log10(1e-2)

    configs = []
    for i in range(n_configs):
        num_layers = int(rng.choice(LAYER_CHOICES))
        configs.append({
            'config_id'    : i,
            'hidden_size'  : int(rng.choice(HIDDEN_CHOICES)),
            'num_layers'   : num_layers,
            # dropout is sampled regardless, but only takes effect when num_layers >= 2
            'dropout'      : float(rng.uniform(0.0, 0.5)) if num_layers >= 2 else 0.0,
            'learning_rate': float(10 ** rng.uniform(LR_LOG_LO, LR_LOG_HI)),
            'batch_size'   : int(rng.choice(BATCH_CHOICES)),
        })
    return configs


# Generate and inspect the search list
configs = sample_configs(N_CONFIGS)
configs_df = pd.DataFrame(configs)
print(f"Generated {len(configs)} configurations.\n")
print(configs_df.head(10).to_string(index=False))
print(f"\nLearning-rate spread (log10):")
print(f"  min = {np.log10(configs_df.learning_rate).min():.2f}, "
      f"max = {np.log10(configs_df.learning_rate).max():.2f}")


Generated 30 configurations.

 config_id  hidden_size  num_layers  dropout  learning_rate  batch_size
         0          256           1 0.000000       0.000755         128
         1           32           2 0.047089       0.008938         256
         2          256           2 0.393032       0.000180         512
         3          128           1 0.000000       0.007137         128
         4          128           2 0.411381       0.000771         128
         5           32           1 0.000000       0.000134         256
         6          256           2 0.315832       0.003282         256
         7           32           1 0.000000       0.006113         512
         8          256           2 0.097319       0.000858         128
         9          128           1 0.000000       0.002323          64

Learning-rate spread (log10):
  min = -3.99, max = -2.05


## 6. Search Runner — Resumable Loop

For each `(config_id, fold)` not already in the results CSV, build the loaders
for that fold, train with early stopping, and append a result row. The CSV
is written incrementally so a Colab disconnect doesn't lose progress.

The structure is `(config × fold) -> row`, not `config -> row` — this means
even partial config evaluations (e.g. 3 of 5 folds completed) are preserved.
We aggregate to per-config means in the analysis cell.


In [6]:
RESULTS_CSV = os.path.join(RESULTS_DIR, f'search_{TUNE_DS}.csv')

RESULT_COLUMNS = [
    'config_id', 'fold',
    'hidden_size', 'num_layers', 'dropout', 'learning_rate', 'batch_size',
    'val_mse', 'val_rmse', 'n_epochs', 'wall_time_sec',
]


def load_existing_results():
    """Return DataFrame of already-evaluated (config_id, fold) rows."""
    if os.path.exists(RESULTS_CSV):
        return pd.read_csv(RESULTS_CSV)
    return pd.DataFrame(columns=RESULT_COLUMNS)


def append_result(row):
    """Atomically append a single row to the CSV, creating header if needed."""
    df = pd.DataFrame([row])[RESULT_COLUMNS]
    write_header = not os.path.exists(RESULTS_CSV)
    df.to_csv(RESULTS_CSV, mode='a', header=write_header, index=False)


def get_completed_pairs():
    """Set of (config_id, fold) tuples already evaluated."""
    df = load_existing_results()
    if df.empty:
        return set()
    return set(zip(df['config_id'].astype(int), df['fold'].astype(int)))


def evaluate_one_fold(config, train_idx, val_idx, full_dataset, n_features):
    """Train one fold of a config; return (val_mse, n_epochs, wall_time)."""
    set_seed(SEED + config['config_id'])  # different init per config, deterministic per (config_id, fold)

    train_ds = Subset(full_dataset, train_idx)
    val_ds   = Subset(full_dataset, val_idx)
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'], shuffle=False)

    model = VanillaLSTM(
        input_size  = n_features,
        hidden_size = config['hidden_size'],
        num_layers  = config['num_layers'],
        dropout     = config['dropout'],
    ).to(DEVICE)

    t0 = time.time()
    val_mse, n_epochs, _ = train_with_early_stopping(
        model, train_loader, val_loader,
        learning_rate = config['learning_rate'],
        max_epochs    = SEARCH_EPOCHS,
        patience      = EARLY_STOP_PATIENCE,
    )
    wall = time.time() - t0
    return val_mse, n_epochs, wall


def run_search(configs, arrays, n_folds=N_FOLDS):
    """Main search loop. Resumes from existing CSV if present."""
    full_dataset = CMAPSSDataset(arrays['X_train'], arrays['y_train'])
    engine_ids   = arrays['engine_ids']
    n_features   = arrays['X_train'].shape[2]

    # Fold splits are fixed by GroupKFold (deterministic given engine_ids)
    kfold = GroupKFold(n_splits=n_folds)
    fold_splits = list(kfold.split(X=np.zeros(len(engine_ids)), groups=engine_ids))

    completed = get_completed_pairs()
    n_total   = len(configs) * n_folds
    n_done    = len(completed)
    print(f"Resuming search: {n_done}/{n_total} (config, fold) evaluations already complete.\n")

    for config in configs:
        cid = config['config_id']
        for fold_idx, (tr_idx, va_idx) in enumerate(fold_splits):
            if (cid, fold_idx) in completed:
                continue

            val_mse, n_epochs, wall = evaluate_one_fold(
                config, tr_idx, va_idx, full_dataset, n_features)

            row = {
                **config,
                'fold'         : fold_idx,
                'val_mse'      : val_mse,
                'val_rmse'     : float(np.sqrt(val_mse)),
                'n_epochs'     : n_epochs,
                'wall_time_sec': wall,
            }
            append_result(row)
            n_done += 1

            print(f"  [{n_done:3d}/{n_total}] cfg={cid:2d} fold={fold_idx} | "
                  f"hs={config['hidden_size']:3d} nl={config['num_layers']} "
                  f"do={config['dropout']:.2f} lr={config['learning_rate']:.1e} "
                  f"bs={config['batch_size']:3d} | "
                  f"val_rmse={np.sqrt(val_mse):6.2f} ep={n_epochs:2d} t={wall:5.1f}s")

    print(f"\n✓ Search complete. Results in {RESULTS_CSV}")


## 7. Run the Search

This is the long-running cell. On a Colab T4 GPU expect ~75 minutes total for
30 configs × 5 folds at ~30 sec/fold. Safe to interrupt and re-run.


In [ ]:
run_search(configs, arrays, n_folds=N_FOLDS)


## 8. Analysis of Search Results

Aggregate per-fold rows into per-config means, find the best config by mean
val RMSE, and visualise the spread across the 30 configurations.


In [ ]:
results_df = pd.read_csv(RESULTS_CSV)
print(f"Loaded {len(results_df)} fold-level rows for {results_df['config_id'].nunique()} configs.")

# Aggregate to per-config: mean and std of val_rmse across folds
agg = results_df.groupby('config_id').agg(
    hidden_size   = ('hidden_size',   'first'),
    num_layers    = ('num_layers',    'first'),
    dropout       = ('dropout',       'first'),
    learning_rate = ('learning_rate', 'first'),
    batch_size    = ('batch_size',    'first'),
    val_rmse_mean = ('val_rmse',      'mean'),
    val_rmse_std  = ('val_rmse',      'std'),
    n_epochs_mean = ('n_epochs',      'mean'),
    n_folds       = ('fold',          'count'),
).reset_index().sort_values('val_rmse_mean')

# Drop incomplete configs (in case search was interrupted mid-config)
complete_mask = agg['n_folds'] == N_FOLDS
if (~complete_mask).any():
    print(f"Note: {(~complete_mask).sum()} configs are incomplete; excluding from analysis.")
agg = agg[complete_mask].reset_index(drop=True)

print(f"\nTop 5 configurations by mean val RMSE:")
print(agg.head().to_string(index=False))

best_row    = agg.iloc[0]
best_config = {
    'config_id'    : int(best_row['config_id']),
    'hidden_size'  : int(best_row['hidden_size']),
    'num_layers'   : int(best_row['num_layers']),
    'dropout'      : float(best_row['dropout']),
    'learning_rate': float(best_row['learning_rate']),
    'batch_size'   : int(best_row['batch_size']),
}
print(f"\nBest config (id={best_config['config_id']}):")
for k, v in best_config.items():
    if k != 'config_id':
        print(f"  {k:14s} = {v}")
print(f"\nMean val RMSE  : {best_row['val_rmse_mean']:.3f}  (± {best_row['val_rmse_std']:.3f} across folds)")


### 8.1 Distribution of search results

Three plots that together answer "did the search find something meaningful, or
did all configs perform similarly?"

1. **Sorted bar chart** of mean val RMSE per config — shows the spread.
2. **Marginal scatter** of val_rmse vs each individual hyperparameter — hints
   at which knobs matter.


In [ ]:
# Plot 1 — sorted bar chart
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(agg)), agg['val_rmse_mean'],
       yerr=agg['val_rmse_std'], color='steelblue', alpha=0.8,
       error_kw={'elinewidth': 0.6, 'alpha': 0.6})
ax.set_xlabel('Config rank')
ax.set_ylabel('Mean val RMSE')
ax.set_title(f'{TUNE_DS} — Random search results, sorted (error bars: std across {N_FOLDS} folds)')
ax.axhline(agg['val_rmse_mean'].iloc[0], color='crimson', linestyle='--',
           linewidth=0.8, label=f'best = {agg["val_rmse_mean"].iloc[0]:.2f}')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{TUNE_DS}_search_sorted.png'), dpi=150,
            bbox_inches='tight')
plt.show()

# Plot 2 — marginals
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
hp_names = ['hidden_size', 'num_layers', 'dropout', 'learning_rate', 'batch_size']
for ax, hp in zip(axes, hp_names):
    if hp == 'learning_rate':
        ax.scatter(agg[hp], agg['val_rmse_mean'], color='steelblue', alpha=0.7)
        ax.set_xscale('log')
    else:
        ax.scatter(agg[hp], agg['val_rmse_mean'], color='steelblue', alpha=0.7)
    ax.set_xlabel(hp)
    ax.set_ylabel('Mean val RMSE')
    ax.grid(True, alpha=0.3)
fig.suptitle(f'{TUNE_DS} — Marginal effect of each hyperparameter on val RMSE',
             y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{TUNE_DS}_search_marginals.png'), dpi=150,
            bbox_inches='tight')
plt.show()


## 9. Final Retrain — Best Config

The best configuration is now retrained on the full 90% engine-grouped train
split (single hold-out, mirroring `Vanilla_LSTM.ipynb`'s split strategy) for
up to 100 epochs with early stopping. This uses *more* training data than any
single CV fold (90% vs 80%) and a longer training budget, so the final test
metrics are slightly better and directly comparable to other CMAPSS papers.

We then evaluate on the held-out test set and report both **RMSE** and the
asymmetric **NASA score**.


In [ ]:
def nasa_score(y_true, y_pred):
    """
    NASA scoring function from Saxena et al. (2008). Asymmetric: late
    predictions (y_pred > y_true) penalised more heavily than early ones.
    """
    e = y_pred - y_true
    s = np.where(e < 0, np.exp(-e / 13.0) - 1.0, np.exp(e / 10.0) - 1.0)
    return float(s.sum())


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


# ── Build engine-grouped 90/10 split (same protocol as Vanilla_LSTM.ipynb) ───
engine_ids   = arrays['engine_ids']
splitter     = GroupShuffleSplit(n_splits=1, test_size=FINAL_VAL_FRACTION,
                                 random_state=SEED)
tr_idx, va_idx = next(splitter.split(X=np.zeros(len(engine_ids)),
                                     groups=engine_ids))

full_dataset = CMAPSSDataset(arrays['X_train'], arrays['y_train'])
test_dataset = CMAPSSDataset(arrays['X_test'],  arrays['y_test'])

train_loader = DataLoader(Subset(full_dataset, tr_idx),
                          batch_size=best_config['batch_size'], shuffle=True)
val_loader   = DataLoader(Subset(full_dataset, va_idx),
                          batch_size=best_config['batch_size'], shuffle=False)
test_loader  = DataLoader(test_dataset,
                          batch_size=best_config['batch_size'], shuffle=False)


# ── Retrain ──────────────────────────────────────────────────────────────────
set_seed(SEED)
final_model = VanillaLSTM(
    input_size  = N_FEATURES,
    hidden_size = best_config['hidden_size'],
    num_layers  = best_config['num_layers'],
    dropout     = best_config['dropout'],
).to(DEVICE)

print(f"Retraining best config (id={best_config['config_id']}) for up to "
      f"{FINAL_EPOCHS} epochs with patience {FINAL_PATIENCE}...")
t0 = time.time()
best_val_mse, n_epochs_run, best_state = train_with_early_stopping(
    final_model, train_loader, val_loader,
    learning_rate = best_config['learning_rate'],
    max_epochs    = FINAL_EPOCHS,
    patience      = FINAL_PATIENCE,
)
print(f"  Done in {time.time()-t0:.1f}s, {n_epochs_run} epochs, "
      f"best val MSE = {best_val_mse:.3f} (val RMSE = {np.sqrt(best_val_mse):.3f})")


# ── Evaluate on test set ─────────────────────────────────────────────────────
final_model.load_state_dict(best_state)
final_model.eval()

y_true_test, y_pred_test = [], []
with torch.no_grad():
    for X, y in test_loader:
        pred = final_model(X.to(DEVICE)).squeeze(-1).cpu().numpy()
        y_pred_test.append(pred)
        y_true_test.append(y.numpy())
y_true_test = np.concatenate(y_true_test)
y_pred_test = np.concatenate(y_pred_test)

test_rmse  = rmse(y_true_test, y_pred_test)
test_score = nasa_score(y_true_test, y_pred_test)


# ── Save artifacts ───────────────────────────────────────────────────────────
final_path = os.path.join(MODELS_DIR, f'VanillaLSTM_{TUNE_DS}_tuned.pt')
torch.save({
    'state_dict' : best_state,
    'config'     : best_config,
    'val_rmse'   : float(np.sqrt(best_val_mse)),
    'test_rmse'  : test_rmse,
    'test_score' : test_score,
    'n_epochs'   : n_epochs_run,
}, final_path)
print(f"\nSaved retrained model -> {final_path}")


# ── Summary table ────────────────────────────────────────────────────────────
print(f"\n{'='*60}\n  Final results — {TUNE_DS} (tuned Vanilla LSTM)\n{'='*60}")
for k, v in best_config.items():
    if k != 'config_id':
        print(f"  {k:14s} = {v}")
print(f"\n  Search-time mean val RMSE : {best_row['val_rmse_mean']:.3f}  "
      f"(± {best_row['val_rmse_std']:.3f})")
print(f"  Final retrained val RMSE  : {np.sqrt(best_val_mse):.3f}")
print(f"  Final test RMSE           : {test_rmse:.3f}")
print(f"  Final test NASA score     : {test_score:.1f}")


### 9.1 Final test predictions — scatter

Predicted vs true RUL on the held-out test set, after retraining the best
configuration. The diagonal is `y = x`. Spread along the diagonal indicates
the model has learnt the temporal signal (compare to the diagnostic in
`Vanilla_LSTM.ipynb` §7.1).


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax_max = max(y_true_test.max(), y_pred_test.max(), 130)

ax.scatter(y_true_test, y_pred_test, s=25, alpha=0.7, color='steelblue',
           edgecolors='white', linewidth=0.5)
ax.plot([0, ax_max], [0, ax_max], color='black', linestyle='--', linewidth=0.8,
        label='y = x')
ax.set_xlim(0, ax_max)
ax.set_ylim(0, ax_max)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('True RUL')
ax.set_ylabel('Predicted RUL')
ax.set_title(f'{TUNE_DS} — Tuned Vanilla LSTM | Test RMSE = {test_rmse:.2f}, '
             f'Score = {test_score:.0f}')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{TUNE_DS}_tuned_test_scatter.png'),
            dpi=150, bbox_inches='tight')
plt.show()
